# Notebook 1 — Environment Setup & Smoke Test

Run this notebook first to install dependencies, mount Google Drive, and verify the Atari environments load correctly.

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/RL_Models'
    IN_COLAB = True
except ImportError:
    import os
    SAVE_DIR = 'models'
    IN_COLAB = False

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

In [ ]:
# Install packages
if IN_COLAB:
    !pip uninstall -y gym
    !pip install -U 'gymnasium[atari]' 'stable-baselines3[extra]'

In [ ]:
import gymnasium as gym
import ale_py

# Register all ALE/Atari environments
gym.register_envs(ale_py)

atari_envs = [e for e in gym.envs.registry.keys() if e.startswith('ALE/')]
print(f'Number of ALE environments registered: {len(atari_envs)}')
print('Sample:', atari_envs[:10])

In [ ]:
# Smoke test each target environment
TARGET_ENVS = ['ALE/Pong-v5', 'ALE/Breakout-v5', 'ALE/SpaceInvaders-v5']

for env_id in TARGET_ENVS:
    env = gym.make(env_id)
    obs, info = env.reset()
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, info = env.step(action)
    print(f'{env_id}: obs shape={obs.shape}, action_space={env.action_space}, reward={reward}')
    env.close()

In [ ]:
# Test make_atari_env + VecFrameStack (standard SB3 preprocessing)
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

vec_env = make_atari_env('ALE/Pong-v5', n_envs=4, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)

obs = vec_env.reset()
print(f'Vectorized obs shape: {obs.shape}')  # (4, 84, 84, 4)
vec_env.close()
print('Setup complete.')